# 02 — Data Validation
**Source:** `data/silver/` (clean layer output)

**Goal:** Validate that the cleaning pipeline produced correct, trustworthy data before any analysis or modelling.

Sections:
1. Load & Schema Check
2. Completeness (Missing Values)
3. Validity Rules (Business Logic)
4. Consistency Checks
5. Duplicate Detection
6. Outlier Audit
7. Validation Summary

## 1 · Load & Schema Check

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Load latest silver file ───────────────────────────────────────────────────
BASE_DIR    = os.path.dirname(os.getcwd())
SILVER_DIR  = os.path.join(BASE_DIR, "data", "silver")
silver_files = sorted(glob.glob(os.path.join(SILVER_DIR, "avito_clean_*.csv")))
assert silver_files, f"No silver files found in {SILVER_DIR}"

latest_file = silver_files[-1]
print(f"Loading: {os.path.basename(latest_file)}")

df = pd.read_csv(latest_file)
print(f"Shape  : {df.shape}")
df.head(3)

In [ ]:
# ── Expected columns ─────────────────────────────────────────────────────────
EXPECTED_COLS = [
    "prix", "prix_type", "ville", "quartier", "surface_m2",
    "nb_chambres", "nb_salles_bain", "etage", "lien", "scraped_at",
    "prix_par_m2", "categorie_prix", "region_label", "is_grande_ville",
]

missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
extra_cols   = [c for c in df.columns if c not in EXPECTED_COLS + ["id", "titre", "annee_construction", "age_bien", "loaded_at"]]

if missing_cols:
    print(f"❌ Missing columns: {missing_cols}")
else:
    print("✅ All expected columns present")

if extra_cols:
    print(f"ℹ️  Extra columns: {extra_cols}")

print("\nActual dtypes:")
print(df.dtypes)

## 2 · Completeness (Missing Values)

In [ ]:
# ── Critical columns: must have NO nulls ─────────────────────────────────────
CRITICAL_COLS  = ["prix", "ville", "lien"]
IMPORTANT_COLS = ["surface_m2", "nb_chambres", "nb_salles_bain", "prix_par_m2"]

n = len(df)
print(f"Total rows: {n}\n")

print("CRITICAL columns (must be 100% filled):")
for col in CRITICAL_COLS:
    null_count = df[col].isna().sum()
    status = "✅" if null_count == 0 else "❌"
    print(f"  {status} {col:<20}: {n - null_count}/{n} filled  ({null_count} nulls)")

print("\nIMPORTANT columns (target ≥ 80% filled):")
for col in IMPORTANT_COLS:
    null_count = df[col].isna().sum()
    pct = 100 * (n - null_count) / n
    status = "✅" if pct >= 80 else ("⚠️" if pct >= 40 else "❌")
    print(f"  {status} {col:<20}: {n - null_count}/{n} filled  ({pct:.1f}%)")

In [ ]:
# ── Visual: missing values heatmap ───────────────────────────────────────────
viz_cols = [c for c in ["prix", "surface_m2", "nb_chambres", "nb_salles_bain",
                         "prix_par_m2", "age_bien", "annee_construction"] if c in df.columns]

fig, ax = plt.subplots(figsize=(11, 3))
sns.heatmap(
    df[viz_cols].isna().T,
    cbar=False, cmap=["#2ecc71", "#e74c3c"],
    linewidths=0.4, ax=ax, yticklabels=viz_cols,
)
ax.set_title("Missing Values Map  (red = null, green = filled)", fontsize=12, fontweight="bold")
ax.set_xlabel("Row index")
plt.tight_layout()
plt.show()

## 3 · Validity Rules (Business Logic)

In [ ]:
VALID_PRIX_TYPES = {"mensuel", "journalier", "inconnu"}
VALID_CATEGORIES = {"Très Bas", "Bas", "Moyen", "Élevé", "Luxe", "Inconnu"}

rules = {
    "prix > 0":
        df["prix"].notna() & (df["prix"] <= 0),

    "prix ≤ 500 000 DH (mensuel)":
        (df.get("prix_type", "mensuel") == "mensuel") & df["prix"].notna() & (df["prix"] > 500_000),

    "surface_m2 > 0":
        df["surface_m2"].notna() & (df["surface_m2"] <= 0),

    "surface_m2 ≤ 5000 m²":
        df["surface_m2"].notna() & (df["surface_m2"] > 5000),

    "nb_chambres in [1, 20]":
        df["nb_chambres"].notna() & ((df["nb_chambres"] < 1) | (df["nb_chambres"] > 20)),

    "nb_salles_bain in [1, 10]":
        df["nb_salles_bain"].notna() & ((df["nb_salles_bain"] < 1) | (df["nb_salles_bain"] > 10)),

    "prix_type is valid":
        ~df["prix_type"].isin(VALID_PRIX_TYPES),

    "categorie_prix is valid":
        df["categorie_prix"].notna() & ~df["categorie_prix"].isin(VALID_CATEGORIES),

    "lien starts with avito.ma":
        ~df["lien"].str.startswith("https://www.avito.ma"),

    "ville not empty or null":
        df["ville"].isna() | (df["ville"].str.strip() == ""),

    "prix_par_m2 consistent (prix / surface_m2)":
        df["prix_par_m2"].notna() & df["prix"].notna() & df["surface_m2"].notna() &
        ((df["prix"] / df["surface_m2"] - df["prix_par_m2"]).abs() > 1),
}

print("Business Rule Validation:\n")
total_violations = 0
for rule_name, fail_mask in rules.items():
    violations = int(fail_mask.sum())
    total_violations += violations
    status = "✅" if violations == 0 else "❌"
    print(f"  {status} {rule_name:<50}: {violations} violation(s)")

print(f"\nTotal violations: {total_violations}")

In [ ]:
# ── Show offending rows ───────────────────────────────────────────────────────
display_cols = [c for c in ["titre", "prix", "prix_type", "ville",
                              "surface_m2", "nb_chambres", "nb_salles_bain",
                              "categorie_prix"] if c in df.columns]

for rule_name, fail_mask in rules.items():
    bad_rows = df[fail_mask]
    if len(bad_rows) > 0:
        print(f"\n❌ Violations — '{rule_name}':")
        print(bad_rows[display_cols].to_string())

## 4 · Consistency Checks

In [ ]:
# ── Check 1: region_label matches expected city-to-region mapping ─────────────
CITY_REGION_MAP = {
    "Casablanca" : "Casablanca-Settat",
    "Rabat"      : "Rabat-Salé-Kénitra",
    "Marrakech"  : "Marrakech-Safi",
    "Fès"        : "Fès-Meknès",
    "Tanger"     : "Tanger-Tétouan-Al Hoceïma",
}

region_errors = 0
for city, expected_region in CITY_REGION_MAP.items():
    wrong = df[(df["ville"] == city) & (df["region_label"] != expected_region)]
    if len(wrong) > 0:
        print(f"❌ {city}: expected '{expected_region}', found {wrong['region_label'].unique()}")
        region_errors += len(wrong)

if region_errors == 0:
    print("✅ All known city-to-region mappings are consistent")

# ── Check 2: is_grande_ville flag is correct ──────────────────────────────────
GRANDES_VILLES = {"Casablanca","Rabat","Marrakech","Fès","Tanger",
                  "Agadir","Meknès","Oujda","Kénitra","Tétouan"}

gv_errors = df[
    (df["ville"].isin(GRANDES_VILLES)  & (df["is_grande_ville"] == False)) |
    (~df["ville"].isin(GRANDES_VILLES) & (df["is_grande_ville"] == True))
]

if len(gv_errors) > 0:
    print(f"\n❌ is_grande_ville errors: {len(gv_errors)} row(s)")
    print(gv_errors[["ville", "is_grande_ville"]].to_string())
else:
    print("\n✅ is_grande_ville flags are all correct")

# ── Check 3: scraped_at is in the past ────────────────────────────────────────
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")
future_dates = df[df["scraped_at"] > pd.Timestamp.now()]

if len(future_dates) > 0:
    print(f"\n❌ Future scraped_at dates: {len(future_dates)} row(s)")
else:
    print("\n✅ All scraped_at timestamps are in the past")

## 5 · Duplicate Detection

In [ ]:
# ── Exact duplicates on lien ──────────────────────────────────────────────────
dup_lien = df[df.duplicated(subset=["lien"], keep=False)]
if len(dup_lien) > 0:
    print(f"❌ Duplicate liens: {len(dup_lien)} rows")
    print(dup_lien[["titre", "lien"]].to_string())
else:
    print("✅ No duplicate liens")

# ── Near-duplicates: same ville + prix + surface_m2 ───────────────────────────
near_dup_cols = [c for c in ["ville", "prix", "surface_m2"] if c in df.columns]
near_dups = df[df.duplicated(subset=near_dup_cols, keep=False)]

if len(near_dups) > 0:
    print(f"\n⚠️  Near-duplicates (same ville+prix+surface_m2): {len(near_dups)} rows")
    print(near_dups[["titre", "ville", "prix", "surface_m2", "lien"]].to_string())
else:
    print("\n✅ No near-duplicates detected")

## 6 · Outlier Audit

In [ ]:
OUTLIER_COLS = [c for c in ["prix", "surface_m2", "prix_par_m2"] if c in df.columns]

print("IQR Outlier Audit (1.5×IQR rule):\n")
for col in OUTLIER_COLS:
    s = df[col].dropna()
    if len(s) < 4:
        print(f"  ⚠️  {col}: insufficient data")
        continue
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col].notna()) & ((df[col] < lo) | (df[col] > hi))]
    pct = 100 * len(outliers) / len(df)
    print(f"  {col:<20}: expected [{lo:,.0f} – {hi:,.0f}]  → {len(outliers)} outliers ({pct:.1f}%)")
    if len(outliers) > 0:
        show_cols = [c for c in ["titre", "ville", col] if c in df.columns]
        print(outliers[show_cols].to_string())
        print()

# ── Boxplots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(OUTLIER_COLS), figsize=(5 * len(OUTLIER_COLS), 4))
if len(OUTLIER_COLS) == 1:
    axes = [axes]
for ax, col in zip(axes, OUTLIER_COLS):
    sns.boxplot(y=df[col].dropna(), ax=ax, color="#4C72B0", width=0.4)
    ax.set_title(col, fontweight="bold")
    sns.despine(ax=ax)
plt.suptitle("Outlier Distribution (IQR boxplot)", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7 · Validation Summary

In [ ]:
scorecard = []

for col in CRITICAL_COLS:
    null_count = int(df[col].isna().sum())
    scorecard.append({"Check": f"No nulls in '{col}'", "Pass": null_count == 0, "Detail": f"{null_count} nulls"})

for rule_name, fail_mask in rules.items():
    v = int(fail_mask.sum())
    scorecard.append({"Check": rule_name, "Pass": v == 0, "Detail": f"{v} violations"})

scorecard.append({"Check": "No duplicate liens", "Pass": len(dup_lien) == 0, "Detail": f"{len(dup_lien)} duplicates"})
scorecard.append({"Check": "Region labels consistent", "Pass": region_errors == 0, "Detail": f"{region_errors} errors"})
scorecard.append({"Check": "is_grande_ville correct", "Pass": len(gv_errors) == 0, "Detail": f"{len(gv_errors)} errors"})

sc_df = pd.DataFrame(scorecard)
sc_df["Status"] = sc_df["Pass"].map({True: "✅ PASS", False: "❌ FAIL"})

passed = int(sc_df["Pass"].sum())
total  = len(sc_df)

print(f"\n{'='*55}")
print(f"  VALIDATION SCORECARD: {passed}/{total} checks passed")
print(f"{'='*55}\n")
print(sc_df[["Status", "Check", "Detail"]].to_string(index=False))

print()
if passed == total:
    print("🎉 Data is CLEAN — safe to proceed to feature engineering.")
else:
    print(f"⚠️  {total - passed} check(s) failed — review before modelling.")